# Módulo 2 · Auditor de políticas financieras con RAG sobre documentos no estructurados
**Caso:** un **Agente Auditor** que, dado un gasto, busca en las normativas internas, se queda con los **3 fragmentos más relevantes** y **justifica la aprobación o el rechazo** citando fuentes reales.

## Parte 0 · Preparación

In [ ]:
import os, sys, json, re, time
from pathlib import Path

# 1) Ubica la raíz del proyecto y prepara los imports
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "config.py").exists())
for ruta in (RAIZ, RAIZ / "modulo2_rag_reranking" / "src", RAIZ / "modulo1_adk_a2a" / "src"):
    sys.path.insert(0, str(ruta))
os.chdir(RAIZ)

# 2) Variables de entorno (.env)
from dotenv import load_dotenv
load_dotenv(RAIZ / ".env")

os.environ["AGENTS_MODE"] = "cloud"
os.environ["GOOGLE_GENAI_USE_ENTERPRISE"] = "TRUE"
os.environ.setdefault("GOOGLE_CLOUD_LOCATION", "us-central1")

import logging, warnings
logging.getLogger("google_adk").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning)

faltan = []
for modulo, paquete in [("numpy", "numpy"), ("sklearn", "scikit-learn"),
                        ("google.cloud.discoveryengine_v1", "google-cloud-discoveryengine")]:
    try:
        __import__(modulo)
    except ImportError:
        faltan.append(paquete)
if faltan:
    raise RuntimeError("Faltan paquetes. En una terminal (con el venv activo) ejecuta:\n"
                       f"    pip install {' '.join(faltan)}\nluego reinicia el kernel de este notebook.")

print("RAIZ:", RAIZ)

In [ ]:
proyecto = os.getenv("GOOGLE_CLOUD_PROJECT", "")
if not proyecto or proyecto == "tu-proyecto":
    raise RuntimeError("Falta GOOGLE_CLOUD_PROJECT en tu .env: pon el ID de tu proyecto (no 'tu-proyecto').")

import google.auth
try:
    google.auth.default()
except google.auth.exceptions.DefaultCredentialsError:
    raise RuntimeError("No hay credenciales. En una terminal ejecuta: gcloud auth application-default login")

# La configuración del reranker se deriva de tu proyecto (si en el .env quedó el valor de ejemplo, se reemplaza)
if "tu-proyecto" in os.getenv("RANKING_CONFIG", "tu-proyecto"):
    os.environ["RANKING_CONFIG"] = f"projects/{proyecto}/locations/global/rankingConfigs/default_ranking_config"

print("Proyecto:", proyecto, "| Región Gemini:", os.environ["GOOGLE_CLOUD_LOCATION"])
print("Ranking config:", os.environ["RANKING_CONFIG"])

In [ ]:
from google import genai
from google.genai import types

cliente = genai.Client()
candidatos = list(dict.fromkeys([os.getenv("EMBEDDING_MODEL", "text-embedding-005"), "gemini-embedding-001"]))
elegido = None
for m in candidatos:
    try:
        r = cliente.models.embed_content(model=m, contents=["prueba uno", "prueba dos"],
                                         config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"))
        if len(r.embeddings) == 2:
            elegido = m
            print(f"✅ {m} → dimensión {len(r.embeddings[0].values)}")
            break
        print(f"⚠️ {m} devolvió {len(r.embeddings)} vectores para 2 textos (no sirve para lotes)")
    except Exception as e:
        print(f"❌ {m} → {str(e)[:160]}")
if not elegido:
    raise RuntimeError("Ningún modelo de embeddings respondió. Revisa el proyecto, la región y que la API de Vertex AI esté habilitada.")
os.environ["EMBEDDING_MODEL"] = elegido

# Ahora sí: configuración compartida del proyecto
from shared.config import MODEL_NAME, RETRIEVE_K, RERANK_N, POLICIES_DIR, DATA_DIR
from finanzas_agents.eventos import preguntar     
MODELO = MODEL_NAME
print("Embeddings:", elegido, "| Modelo de texto:", MODELO, "| K =", RETRIEVE_K, "| N =", RERANK_N)

## Parte 1 · Los documentos: qué tienen de «no estructurado» 
Un documento **estructurado** trae marcas que un programa puede seguir: en nuestras políticas en markdown, cada artículo empieza con `## Artículo 1.3 – Autorización previa`. Basta cortar ahí.

Un **PDF** no trae esas marcas de forma utilizable, y un **escaneo** ni siquiera tiene texto: es una imagen de texto. Antes de buscar nada hay que **leerlo**.

Usamos PDFs **sintéticos** (empresa y cifras ficticias):
| Carpeta | Archivo | Qué es |
|---|---|---|
| `data/politicas_pdf/` | `POL-VIA`, `POL-CON`, `POL-PRE`, `POL-COB` | Las 4 políticas, **con texto** |
| `data/politicas_pdf/` | `AUD-2026` | Las notas de auditoría, **escaneadas** (solo imagen) |
| `data/reportes/` | `10q_norte_3t26` | El reporte trimestral 10-Q, con texto y una tabla |
| `data/reportes/` | `10q_norte_3t26_escaneado` | El mismo reporte, **escaneado** |

In [ ]:
PDF_POLITICAS = DATA_DIR / "politicas_pdf"
PDF_REPORTES = DATA_DIR / "reportes"
for carpeta in (PDF_POLITICAS, PDF_REPORTES):
    for f in sorted(carpeta.glob("*.pdf")):
        tipo = "escaneado (solo imagen)" if ("escaneado" in f.name or f.name.startswith("AUD")) else "con texto"
        print(f"{f.parent.name:14} {f.name:30} {f.stat().st_size:>8,} bytes   {tipo}")

## Parte 2 · Leer los PDFs: analizador por defecto contra analizador de LLM 
Usamos **RAG Engine** en su modo **sin servidores**: sube los archivos desde Cloud Storage, los lee, los corta, calcula los embeddings y los guarda en **Vector Search** administrado. Para leer PDFs ofrece dos formas, y las comparamos con **el mismo reporte 10-Q** importado a dos corpus:

| Corpus | Analizador | Qué esperamos |
|---|---|---|
| `pdf_defecto` | El de por defecto (gratis) | Extrae el texto del PDF con texto; del escaneado, nada |
| `pdf_llm` | **Analizador de LLM** (Gemini) | Lee también el escaneado y conserva la tabla |

El **Layout Parser de Document AI** es la tercera opción documentada; necesita crear un procesador aparte y cobra por página. **No lo ejecutamos.**

### Antes de empezar (una sola vez por proyecto)
- **APIs habilitadas:** Vertex AI, **Discovery Engine** (Ranking API), **Vector Search** y **Cloud Resource Manager** (la usa el SDK al importar desde un bucket).
- **Modo sin servidores** de RAG Engine activo en `us-central1` (la celda 2.1 lo muestra; es del **proyecto**, no tuyo: si no está, avisa al instructor).
- Permiso para crear buckets y corpus. Se usan **las mismas credenciales** del resto del notebook.

In [ ]:
# 2.1 · Preparación: SDK, token para la API REST y modo actual del proyecto
import httpx, subprocess, shutil, tempfile
import google.auth, google.auth.transport.requests
from google.genai import types as genai_types

LOCATION = "us-central1"          # el modo sin servidores solo existe aquí
PROYECTO = proyecto

try:
    import agentplatform
    from agentplatform import types as ap_types
except ImportError:
    raise RuntimeError("No encuentro el módulo 'agentplatform' (el que usa el quickstart de RAG Engine). "
                       "Prueba: pip install --upgrade google-cloud-aiplatform  y reinicia el kernel. "
                       "Si sigue sin existir, el SDK cambió de nombre: revisa la página del quickstart.")

_cred, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])

def rest(metodo: str, ruta: str, **kw):
    """Llamada REST a la API v1beta1 de Agent Platform (la que documentan para el modo de RAG Engine)."""
    _cred.refresh(google.auth.transport.requests.Request())
    url = f"https://{LOCATION}-aiplatform.googleapis.com/v1beta1/{ruta}"
    cab = {"Authorization": f"Bearer {_cred.token}", "Content-Type": "application/json",
           "x-goog-user-project": PROYECTO}
    r = httpx.request(metodo, url, headers=cab, timeout=120, **kw)
    if r.status_code >= 400:
        raise RuntimeError(f"{metodo} {ruta} → HTTP {r.status_code}: {r.text[:400]}")
    return r.json() if r.text else {}

RUTA_CONFIG = f"projects/{PROYECTO}/locations/{LOCATION}/ragEngineConfig"
config_actual = rest("GET", RUTA_CONFIG)
print("Modo actual de RAG Engine en tu proyecto:")
print(json.dumps(config_actual, indent=2, ensure_ascii=False))

# Sin servidores = Vector Search administrado. La forma exacta de la respuesta no está verificada: lee el JSON de arriba.
MODO_SERVERLESS = "serverless" in json.dumps(config_actual).lower()
print("\n→", "modo SIN SERVIDORES (Vector Search administrado) ✅" if MODO_SERVERLESS
      else "⚠️ no veo 'serverless' en la configuración: no sigas sin avisar al instructor")

In [ ]:
# 2.2 · ¿Hay corpus existentes en el proyecto?
lista = rest("GET", f"projects/{PROYECTO}/locations/{LOCATION}/ragCorpora")
corpora = lista.get("ragCorpora", [])
print(len(corpora), "corpus existentes en", LOCATION)
for c in corpora:
    print(" -", c.get("displayName"), "|", c.get("name"))
if not corpora:
    print("Ninguno: cambiar de modo no afecta datos de nadie.")

In [ ]:
# 2.3 · Cambiar al modo sin servidores 
CAMBIAR_A_SERVERLESS = False   
if CAMBIAR_A_SERVERLESS:
    print(json.dumps(rest("PATCH", RUTA_CONFIG, json={"ragManagedDbConfig": {"serverless": {}}}), indent=2))
else:
    print("No se cambió el modo del proyecto.")

In [ ]:
# 2.4 · Un bucket con los PDFs, en dos carpetas: políticas y reportes
import getpass
from google.cloud import storage

if not MODO_SERVERLESS:
    raise RuntimeError("El proyecto no está en modo sin servidores. No sigas: avisa al instructor.")

USUARIO = re.sub(r"[^a-z0-9]", "", getpass.getuser().lower())[:12] or "alumno"      # evita choques entre compañeros
BUCKET = f"{PROYECTO}-rag-m2-{USUARIO}"
CORPUS_CREADOS = []                                                                    # para la limpieza final

gcs = storage.Client(project=PROYECTO)
bucket = gcs.bucket(BUCKET)
if not bucket.exists():
    bucket = gcs.create_bucket(BUCKET, location=LOCATION)
for prefijo, carpeta in (("pdf_politicas", PDF_POLITICAS), ("pdf_reportes", PDF_REPORTES)):
    for f in sorted(carpeta.glob("*.pdf")):
        bucket.blob(f"{prefijo}/{f.name}").upload_from_filename(str(f))
for o in gcs.list_blobs(BUCKET):
    print(f"{o.name:46} {o.size:>9,} bytes")

In [ ]:
# 2.5 · Herramientas: crear un corpus (backend explícito), importar con un analizador y recuperar
def crear_corpus(nombre: str) -> str:
    lista = rest("GET", f"projects/{PROYECTO}/locations/{LOCATION}/ragCorpora").get("ragCorpora", [])
    ya = [c for c in lista if c.get("displayName") == nombre]
    if ya:
        print("Reutilizo:", nombre)
        nombre_recurso = ya[0]["name"]
    else:
        op = rest("POST", f"projects/{PROYECTO}/locations/{LOCATION}/ragCorpora",
                  json={"display_name": nombre, "vector_db_config": {"rag_managed_vertex_vector_search": {}}})
        for _ in range(24):
            op = rest("GET", op["name"])
            if op.get("done"):
                break
            time.sleep(5)
        if "error" in op or not op.get("done"):
            raise RuntimeError("No se creó el corpus: " + json.dumps(op, ensure_ascii=False)[:600])
        print("Creado:", nombre)
        nombre_recurso = op["response"]["name"]
    CORPUS_CREADOS.append(nombre_recurso)
    return nombre_recurso

client = agentplatform.Client(project=PROYECTO, location=LOCATION)
MODELO_PARSER = f"projects/{PROYECTO}/locations/{LOCATION}/publishers/google/models/gemini-2.5-flash"

def parser_llm(prompt: str | None = None):
    kw = dict(model_name=MODELO_PARSER, max_parsing_requests_per_min=30)
    if prompt:
        kw["custom_parsing_prompt"] = prompt
    return ap_types.RagFileParsingConfig(llm_parser=ap_types.RagFileParsingConfigLlmParser(**kw))

def importar(corpus: str, carpeta: str, tamano: int = 512, solape: int = 100, parsing=None) -> None:
    cfg = dict(
        gcs_source=genai_types.GcsSource(uris=[f"gs://{BUCKET}/{carpeta}/"]),        # la carpeta con barra final
        rag_file_transformation_config=ap_types.RagFileTransformationConfig(
            rag_file_chunking_config=ap_types.RagFileChunkingConfig(chunk_size=tamano, chunk_overlap=solape)),
        max_embedding_requests_per_min=1000)
    if parsing is not None:
        cfg["rag_file_parsing_config"] = parsing
    client.rag.import_files(name=corpus, import_config=ap_types.ImportRagFilesConfig(**cfg))

def recuperar_contextos(corpus: str, consulta: str, k: int = RETRIEVE_K, intentos: int = 4) -> list:
    for _ in range(intentos):
        resp = client.rag.retrieve_contexts(
            vertex_rag_store=genai_types.VertexRagStore(
                rag_resources=[genai_types.VertexRagStoreRagResource(rag_corpus=corpus)]),
            query=ap_types.RagQuery(text=consulta, rag_retrieval_config=genai_types.RagRetrievalConfig(top_k=k)))
        ctx = getattr(getattr(resp, "contexts", None), "contexts", None) or []
        if ctx:
            return ctx
        time.sleep(10)
    return []
print("listo")

In [ ]:
# 2.6 · El mismo reporte 10-Q, importado con cada analizador (las operaciones sobre corpus no admiten concurrencia: una tras otra)
c_def = crear_corpus(f"pdf_defecto_{USUARIO}")
c_llm = crear_corpus(f"pdf_llm_{USUARIO}")
for etiqueta, corpus, parsing in (("analizador por defecto", c_def, None), ("analizador de LLM", c_llm, parser_llm())):
    t0 = time.time()
    try:
        importar(corpus, "pdf_reportes", parsing=parsing)
        print(f"✅ importación terminada con el {etiqueta} ({time.time() - t0:.0f} s)")
    except Exception as e:
        print(f"⚠️ {etiqueta}: {type(e).__name__}: {str(e)[:400]}")

In [ ]:
# 2.7 · La misma pregunta a los dos corpus: ¿qué leyó cada analizador?
CONSULTA_PDF = "¿Cuál fue la utilidad neta del 3T 2026 y cuánto aumentó la consultoría?"
PDFS_10Q = ["10q_norte_3t26.pdf", "10q_norte_3t26_escaneado.pdf"]
PRUEBAS = {"utilidad neta (4,015)": "4,015", "consultoría (2,880)": "2,880"}
LEIDO = {}
for etiqueta, corpus in (("analizador por defecto", c_def), ("analizador de LLM", c_llm)):
    ctx = recuperar_contextos(corpus, CONSULTA_PDF, k=20)
    LEIDO[etiqueta] = ctx
    print(f"\n=== {etiqueta}: {len(ctx)} fragmentos recuperados")
    for archivo in PDFS_10Q:
        propios = [c for c in ctx if archivo in (getattr(c, "source_uri", "") or "")]
        hallado = {k: any(v in (c.text or "") for c in propios) for k, v in PRUEBAS.items()}
        marcas = "  ".join(f"{'✅' if ok else '❌'} {k}" for k, ok in hallado.items())
        print(f"  {archivo:30} {len(propios):>2} fragmentos | {marcas}")

In [ ]:
# 2.8 · La tabla, tal como la dejó el analizador de LLM
ctx_llm = LEIDO["analizador de LLM"]
for c in ctx_llm:
    t = c.text or ""
    if "Utilidad neta" not in t:
        continue
    fuente = (getattr(c, "source_uri", "") or "").split("/")[-1]
    ini = t.find("<table")
    print("--- fuente:", fuente)
    print(t[ini:ini + 2500] if ini >= 0 else t[:2500])
    print()
anotaciones = sum((c.text or "").count("Context") for c in ctx_llm)
print(f"Anotaciones «Context» escritas por el modelo en los fragmentos recuperados: {anotaciones}")

> **Qué salió y cómo interpretarlo:**
> - **PDF con texto:** los dos analizadores recuperan las cifras.
> - **PDF escaneado:** el analizador por defecto importa el archivo **sin error y sin guardar nada**: es una **pérdida silenciosa**, nadie avisa de que el documento quedó fuera. El de LLM lee la imagen y recupera las cifras.
> - **La tabla** sale como **HTML** (`<table>`), y el modelo **añade comentarios propios** («Context: …», en inglés) que no están en el documento. Sirven para recuperar mejor, pero es **texto que escribió un modelo**: hay que **verificar las cifras contra el original** (compara la tabla de arriba con el PDF) y se puede controlar el formato con un `custom_parsing_prompt`, que usaremos en la Parte 3.
>
> **Cuándo usar cuál:** el analizador por defecto basta con PDFs con texto simple. El de LLM entra con tablas, imágenes o escaneos, y se paga por página. El Layout Parser de Document AI es la alternativa especializada en estructura, con su propio costo.

## Parte 3 · Cortar el texto: tres formas de chunking
Ya sabemos **leer** los PDFs. Ahora hay que **cortar** el texto en fragmentos: es la decisión que más pesa en la calidad de un RAG. Comparamos **tres formas**, todas guardadas en **Vector Search** con RAG Engine, con las **políticas** como material:

| | Quién corta | Cómo |
|---|---|---|
| **A · RAG Engine** | El **servicio** | Subimos los PDFs tal cual: él los lee y los corta (por tamaño, en tokens) |
| **B · Semántico (nuestro)** | **Nosotros** | Sobre el texto que extrajo el analizador, cortamos donde baja la similitud entre oraciones; subimos cada trozo como un archivo |
| **C · Por estructura (nuestro)** | **Nosotros** | Sobre ese mismo texto, un trozo por «Artículo»; subimos cada trozo como un archivo |

In [ ]:
# 3.1 · Forma A: los PDFs de las políticas, tal cual; el servicio los lee (analizador de LLM con prompt propio) y los corta
PROMPT_PARSER = ("Transcribe el documento fielmente, en español. Conserva los encabezados («Artículo», «Nota», «Item») "
                 "con su numeración y las tablas. No añadas comentarios, resúmenes ni explicaciones.")
c_A = crear_corpus(f"pol_A_{USUARIO}")
t0 = time.time()
importar(c_A, "pdf_politicas", parsing=parser_llm(PROMPT_PARSER))
print(f"✅ A: PDFs importados y cortados por RAG Engine ({time.time() - t0:.0f} s)")

In [ ]:
# 3.2 · De un fragmento de RAG Engine a un «chunk» con documento, artículos y cita
from dataclasses import dataclass
from rag.chunking import Chunk
from rag.retrieval import Scored

@dataclass(frozen=True)
class ChunkDoc(Chunk):
    articulos: tuple = ()          # números de artículo o nota que aparecen dentro del fragmento

    @property
    def citation(self) -> str:
        if self.articulos:
            etiqueta = "Art." if len(self.articulos) == 1 else "Arts."
            return f"{self.doc}, {etiqueta} {' y '.join(self.articulos)} ({self.title or 'sin título'})"
        return f"{self.doc}, fragmento {self.article} ({self.title or 'sin sección'})"

RE_ART = re.compile(r"(?:Art[íi]culo|Nota)\s+(\d+\.\d+)")
RE_TIT = re.compile(r"(?:Art[íi]culo|Nota)\s+(\d+\.\d+)\s*[-–]\s*([^\n*<|]+)")
RE_ITEM = re.compile(r"Item\s+(1A|1|2)\s*[-–]\s*([^\n<*(]+)")

def doc_de(fuente: str) -> str:
    nombre = fuente.split("/")[-1].rsplit(".", 1)[0].split("__")[0]          # «POL-VIA-2026__1.3.md» → POL-VIA-2026
    if nombre.startswith("10q"):
        return "10-Q 3T 2026 (PDF escaneado)" if "escaneado" in nombre else "10-Q 3T 2026 (PDF)"
    return nombre

def a_chunk(c, i: int) -> ChunkDoc:
    texto, fuente = c.text or "", getattr(c, "source_uri", "") or ""
    doc = doc_de(fuente)
    arts = tuple(dict.fromkeys(RE_ART.findall(texto)))
    titulos = {}
    for n, t in RE_TIT.findall(texto):
        titulos.setdefault(n, t.strip())
    if arts:
        titulo, cid, art = titulos.get(arts[0], ""), f"{doc}#{arts[0]}", arts[0]
    else:
        m = RE_ITEM.search(texto)
        titulo = f"Item {m.group(1)} – {m.group(2).strip()}" if m else ""
        cid, art = f"{doc}#frag{i}", str(i + 1)
    return ChunkDoc(id=cid, doc=doc, article=art, title=titulo, text=texto, source=fuente, articulos=arts)

def recuperador(corpus: str):
    # Devuelve la función «consulta → candidatos» de un corpus: es lo ÚNICO que cambia de una fuente a otra.
    def recuperar(consulta: str, k: int = RETRIEVE_K) -> list[Scored]:
        vistos, salida = set(), []
        for i, c in enumerate(recuperar_contextos(corpus, consulta, k=k)):
            clave = " ".join((c.text or "").split())
            if clave not in vistos:
                vistos.add(clave)
                salida.append(Scored(a_chunk(c, i), 0.0))
        return salida
    return recuperar

def todos_los_fragmentos(corpus: str) -> list[ChunkDoc]:
    # Reúne los fragmentos distintos de un corpus con varias consultas generales (para contarlos y mirarlos).
    vistos, salida = set(), []
    for consulta in ("política", "artículo", "gasto", "autorización", "presupuesto", "cobranza", "auditoría"):
        try:
            ctx = recuperar_contextos(corpus, consulta, k=50)
        except Exception:
            ctx = recuperar_contextos(corpus, consulta, k=20)
        for c in ctx:
            clave = " ".join((c.text or "").split())
            if clave not in vistos:
                vistos.add(clave)
                salida.append(a_chunk(c, len(salida)))
    return salida

FRAG_A = todos_los_fragmentos(c_A)
print(f"Forma A: {len(FRAG_A)} fragmentos · anotaciones «Context»: {sum(f.text.count('Context') for f in FRAG_A)}")
for f in sorted(FRAG_A, key=lambda f: (f.doc, f.articulos)):
    print(f"  {f.doc:14} | artículos: {', '.join(f.articulos) or '—':26} | {len(f.text):>5} car.")

In [ ]:
# 3.3 · Tres formas de cortar el MISMO texto extraído (la política de viáticos)
import numpy as np
EMB = os.environ["EMBEDDING_MODEL"]

def embeber(textos: list[str], tarea: str) -> np.ndarray:
    r = cliente.models.embed_content(model=EMB, contents=textos, config=types.EmbedContentConfig(task_type=tarea))
    m = np.array([e.values for e in r.embeddings], dtype=float)
    return m / np.linalg.norm(m, axis=1, keepdims=True)

def oraciones(texto: str) -> list[str]:
    return [s.strip() for s in re.split(r"(?<=[.;])\s+", texto) if s.strip()]

def chunk_semantico(texto: str, percentil: int = 25) -> list[str]:
    ors = oraciones(texto)
    if len(ors) < 3:
        return [texto]
    v = embeber(ors, "RETRIEVAL_DOCUMENT")
    sims = (v[:-1] * v[1:]).sum(axis=1)                  # similitud de cada oración con la siguiente
    umbral = float(np.percentile(sims, percentil))       # se corta en las caídas más fuertes
    grupos, actual = [], [ors[0]]
    for i, s in enumerate(sims):
        if s <= umbral:
            grupos.append(" ".join(actual))
            actual = []
        actual.append(ors[i + 1])
    grupos.append(" ".join(actual))
    return grupos

def por_estructura(texto: str) -> list[str]:
    return [p for p in re.split(r"\n(?=\W*(?:Art[íi]culo|Nota)\s+\d+\.\d+)", texto) if p.strip()]

TEXTOS = {}                                                                  # el texto que extrajo el analizador, por documento
for f in sorted(FRAG_A, key=lambda f: (f.doc, f.articulos)):
    TEXTOS[f.doc] = (TEXTOS.get(f.doc, "") + "\n" + f.text).strip()

DOC = "POL-VIA-2026"
partes = {
    "A · RAG Engine (lo que dejó el servicio)": [f.text for f in FRAG_A if f.doc == DOC],
    "C · por estructura (en cada «Artículo»)": por_estructura(TEXTOS[DOC]),
    "B · semántico (donde baja la similitud)": chunk_semantico(TEXTOS[DOC]),
}
for nombre, trozos in partes.items():
    print(f"\n=== {nombre}: {len(trozos)} trozos")
    for t in trozos:
        arts = ", ".join(dict.fromkeys(RE_ART.findall(t))) or "—"
        print(f"  arts: {arts:20} | {len(t):>4} car. | {' '.join(t.split())[:60]}")

**🧪 Ejercicio 3 · El corte semántico y su percentil.** El método corta donde la similitud entre una oración y la siguiente cae por debajo de un **percentil**: con 25, corta en el cuarto más bajo. Cambia `PERCENTIL` a **10** y a **50**. ¿Qué pasa con el número de trozos? ¿Algún trozo separa el **Art. 1.3** (la regla) de su excepción, el **1.4**? 💸 Una llamada de embeddings.

In [ ]:
# 🧪 Ejercicio 3 · ¿Cuántos trozos salen con otro percentil?
PERCENTIL = 25        # TODO: prueba 10 y 50
trozos = chunk_semantico(TEXTOS[DOC], percentil=PERCENTIL)
print(f"percentil {PERCENTIL}: {len(trozos)} trozos")
for t in trozos:
    print(f"  arts: {', '.join(dict.fromkeys(RE_ART.findall(t))) or '—':20} | {len(t):>4} car.")

In [ ]:
# 3.4 · Formas B y C sobre TODAS las políticas: cada trozo se sube como un archivo y se indexa en su propio corpus
trozos_B, trozos_C = {}, {}
for doc, texto in TEXTOS.items():
    for i, t in enumerate(chunk_semantico(texto)):
        trozos_B[f"{doc}__s{i}"] = t
    for p in por_estructura(texto):
        m = RE_ART.search(p)
        if m:                                                                 # el encabezado del documento no es un artículo
            trozos_C[f"{doc}__{m.group(1)}"] = f"{doc}\n{p.strip()}"

for carpeta, trozos in (("pol_B", trozos_B), ("pol_C", trozos_C)):
    for nombre, texto in trozos.items():
        bucket.blob(f"{carpeta}/{nombre}.md").upload_from_string(texto, content_type="text/markdown")
print(f"B: {len(trozos_B)} archivos · C: {len(trozos_C)} archivos")

c_B = crear_corpus(f"pol_B_{USUARIO}")
c_C = crear_corpus(f"pol_C_{USUARIO}")
for corpus, carpeta, etiqueta in ((c_B, "pol_B", "B"), (c_C, "pol_C", "C")):
    t0 = time.time()
    importar(corpus, carpeta)                       # archivos de texto: el analizador por defecto basta
    print(f"✅ {etiqueta}: importado en {time.time() - t0:.0f} s")

In [ ]:
# 3.5 · Medir las tres formas: fragmentos, artículos por fragmento y Hit@3 sobre los 6 casos
from rag.retrieval import VertexRanker
reranker = VertexRanker(os.environ["RANKING_CONFIG"])
casos = json.loads((DATA_DIR / "casos_gasto.json").read_text(encoding="utf-8"))

def articulos_de(chunk) -> set:
    # Artículos que contiene un chunk. Un fragmento sin artículos (p. ej. del 10-Q) no aporta ninguno.
    return set(chunk.articulos) if hasattr(chunk, "articulos") else {chunk.article}

def tabla_hit3(candidatos_fn, mostrar: bool = True) -> tuple[int, int]:
    # Hit@3 sin y con reranking sobre los 6 casos (acierta si algún artículo esperado está en el top-3).
    ok1 = ok2 = 0
    if mostrar:
        print(f"{'caso':5} {'esperado':9} {'solo etapa 1':30} {'con reranking':30}")
    for c in casos:
        esperado, cand = set(c["articulos_esperados"]), candidatos_fn(c["descripcion"])
        try:
            top = reranker.rerank(c["descripcion"], cand, RERANK_N)
        except Exception as e:
            raise RuntimeError("La Ranking API falló. Revisa que Discovery Engine esté habilitado, tus permisos y RANKING_CONFIG. "
                               f"Detalle: {str(e)[:300]}") from e
        a1 = {a for h in cand[:RERANK_N] for a in articulos_de(h.chunk)}
        a2 = {a for h in top for a in articulos_de(h.chunk)}
        h1, h2 = bool(a1 & esperado), bool(a2 & esperado)
        ok1, ok2 = ok1 + h1, ok2 + h2
        if mostrar:
            print(f"{c['id']:5} {','.join(sorted(esperado)):9} {str(sorted(a1)) + (' ✅' if h1 else ' ❌'):30} "
                  f"{str(sorted(a2)) + (' ✅' if h2 else ' ❌'):30}")
    if mostrar:
        print(f"\nHit@3  sin reranking: {ok1}/{len(casos)}   con reranking: {ok2}/{len(casos)}")
    return ok1, ok2

BRAZOS = {"A": ("RAG Engine (cortes del servicio)", c_A),
          "B": ("semántico, nuestro", c_B),
          "C": ("por estructura, nuestro", c_C)}
RECUP = {k: recuperador(corpus) for k, (_, corpus) in BRAZOS.items()}
N_FRAGS, RESUMEN_BRAZOS = {}, {}
for k, (nombre, corpus) in BRAZOS.items():
    fr = todos_los_fragmentos(corpus)
    N_FRAGS[k] = len(fr)
    arts_medio = sum(len(f.articulos) for f in fr) / max(len(fr), 1)
    RESUMEN_BRAZOS[k] = (tabla_hit3(RECUP[k], mostrar=False), arts_medio, len(fr))

print(f"{'forma':36} {'fragmentos':>10} {'arts/fragmento':>15} {'Hit@3 sin':>10} {'Hit@3 con':>10}")
for k, (nombre, _) in BRAZOS.items():
    (h1, h2), arts_medio, n = RESUMEN_BRAZOS[k]
    print(f"{k + ' · ' + nombre:36} {n:>10} {arts_medio:>15.1f} {f'{h1}/{len(casos)}':>10} {f'{h2}/{len(casos)}':>10}")

# Se elige la forma con mejor Hit@3 con reranking; en empate, la de menos artículos por fragmento (citas más exactas). Puedes forzarla.
ELEGIDO = max(RESUMEN_BRAZOS, key=lambda k: (RESUMEN_BRAZOS[k][0][1], -RESUMEN_BRAZOS[k][1], RESUMEN_BRAZOS[k][2]))
recuperar_pol = RECUP[ELEGIDO]
print(f"\n→ Seguimos con la forma {ELEGIDO} · {BRAZOS[ELEGIDO][0]}")

> **Cómo leer la tabla:** el Hit@3 mide si **algún** artículo esperado llega al top-3. Si un fragmento trae **varios artículos** (forma A), acertar es más fácil, pero la **cita** pierde exactitud («Arts. 1.1 y 1.2 y 1.3…»). Por eso se desempata por **artículos por fragmento**: menos artículos por fragmento significa una cita más precisa. Con un corpus tan pequeño, es normal que varias formas empaten en Hit@3.

## Parte 4 · Indexar: embeddings y Vector Search 
Al importar (los PDFs de la forma A, los trozos de las formas B y C), RAG Engine **calculó los embeddings** de cada fragmento y los guardó en **Vector Search administrado**: crea una colección por corpus en tu proyecto. Es lo que pide el cronograma: *embeddings y almacenamiento en Vector Search*.

La recuperación del siguiente paso (`retrieve_contexts`) consulta esa colección. No hay nada que hacer aquí salvo entender que **el almacén ya es persistente**: sobrevive al cierre del notebook, a diferencia de un índice en memoria.

In [ ]:
import numpy as np
import zlib, struct

def png_barras(valores, ancho=240, alto=140) -> bytes:
    filas = []
    maximo = max(valores)
    for y in range(alto):
        fila = [(255, 255, 255)] * ancho
        for i, v in enumerate(valores):
            x0 = 30 + i * 50
            tope = alto - 10 - int(100 * v / maximo)
            if tope <= y < alto - 10:
                for x in range(x0, x0 + 35):
                    fila[x] = (30, 90, 200)
        if y == alto - 10:
            fila = [(0, 0, 0)] * ancho
        filas.append(bytes([0]) + bytes(c for px in fila for c in px))
    def trozo(tipo, datos):
        return struct.pack(">I", len(datos)) + tipo + datos + struct.pack(">I", zlib.crc32(tipo + datos) & 0xFFFFFFFF)
    return (bytes([137, 80, 78, 71, 13, 10, 26, 10])
            + trozo(b"IHDR", struct.pack(">IIBBBBB", ancho, alto, 8, 2, 0, 0, 0))
            + trozo(b"IDAT", zlib.compress(b"".join(filas)))
            + trozo(b"IEND", b""))

IMG = png_barras([45.1, 46.8, 47.5, 48.2])
MODELO_MM = "gemini-embedding-2"

cliente_mm = genai.Client(enterprise=True, project=proyecto, location="global")

def embeber_mm(contenido) -> np.ndarray:
    v = np.array(cliente_mm.models.embed_content(model=MODELO_MM, contents=contenido).embeddings[0].values, dtype=float)
    return v / np.linalg.norm(v)

try:
    v_img = embeber_mm(types.Part.from_bytes(data=IMG, mime_type="image/png"))
    print(f"✅ {MODELO_MM} respondió: imagen y texto en el mismo espacio ({len(v_img)} dim.)")
    for t in ["gráfico de barras con ingresos trimestrales", "receta de paella valenciana"]:
        print(f"  similitud imagen ↔ «{t}»: {float(v_img @ embeber_mm(t)):.3f}")
except Exception as e:
    print(f"⚠️ No pude usar {MODELO_MM}: {type(e).__name__}: {str(e)[:300]}")
    print("   No es un error de tu notebook: el modelo multimodal puede no estar disponible en tu proyecto.")

## Parte 5 · Recuperar en dos etapas: K = 20 → N = 3 
**Etapa 1:** la búsqueda vectorial de RAG Engine trae los K = 20 fragmentos más cercanos. **Etapa 2:** la **Ranking API** (un modelo que lee la pregunta y cada fragmento juntos) los reordena y nos quedamos con N = 3. K y N vienen del cronograma.

Medimos con los **6 casos de gasto** de `data/casos_gasto.json`: un caso **acierta** si alguno de los artículos esperados aparece en los 3 fragmentos. Seguimos con la **forma de chunking elegida en la Parte 3**.

In [ ]:
# 5.1 · Hit@3 sobre los 6 casos, con la forma de chunking elegida
print(f"Forma elegida: {ELEGIDO} · {BRAZOS[ELEGIDO][0]}\n")
tabla_hit3(recuperar_pol)

In [ ]:
# 5.2 · Un caso donde el reranking cambia el resultado
def comparar(consulta: str) -> None:
    cand = recuperar_pol(consulta)
    top = reranker.rerank(consulta, cand, RERANK_N)
    print(f"Consulta: {consulta}\n\nSolo etapa 1:")
    for h in cand[:RERANK_N]:
        print(f"  {h.chunk.citation}")
    print("\nCon reranking:")
    for h in top:
        print(f"  {h.score:.3f}  {h.chunk.citation}")
    print("\n→", "el reranking CAMBIÓ el top-3" if {h.chunk.id for h in cand[:RERANK_N]} != {h.chunk.id for h in top}
          else "mismo top-3 en ambas etapas")

comparar("Se pagó una cena a un proveedor y no tengo factura")

In [ ]:
# 5.3 · ¿Cuántos candidatos hacen falta? Hit@3 con distintos K
candidatos_por_caso = {c["id"]: recuperar_pol(c["descripcion"]) for c in casos}

def hit3_con_k(k: int) -> int:
    aciertos = 0
    for c in casos:
        top_k = reranker.rerank(c["descripcion"], candidatos_por_caso[c["id"]][:k], RERANK_N)
        aciertos += bool({a for h in top_k for a in articulos_de(h.chunk)} & set(c["articulos_esperados"]))
    return aciertos

for k in (3, 5, 10, 20):
    print(f"K = {k:2} → Hit@3 con reranking: {hit3_con_k(k)}/{len(casos)}")
print(f"(el corpus elegido tiene {N_FRAGS[ELEGIDO]} fragmentos en total)")

### Ventana de contexto: cuánto texto llega al modelo

In [ ]:
# 5.4 · Ventana de contexto: con y sin reranking
consulta_ctx = casos[1]["descripcion"]                                      # G2
cand20 = candidatos_por_caso[casos[1]["id"]]
top3 = reranker.rerank(consulta_ctx, cand20, RERANK_N)
texto20 = "\n\n".join(h.chunk.text for h in cand20)
texto3 = "\n\n".join(h.chunk.text for h in top3)

def tokens(texto: str):
    try:
        return cliente.models.count_tokens(model=MODELO, contents=texto).total_tokens, "exacto"
    except Exception:
        return len(texto) // 4, "aprox. (caracteres / 4)"

t20, modo20 = tokens(texto20)
t3, modo3 = tokens(texto3)
print(f"Con los {len(cand20)} candidatos: {len(texto20):>6,} caracteres · {t20:>5,} tokens ({modo20})")
print(f"Con los {len(top3)} del reranking: {len(texto3):>6,} caracteres · {t3:>5,} tokens ({modo3})")
print(f"→ el modelo recibe {100 * (1 - t3 / t20):.0f} % menos contexto")

## Parte 6 · Umbral de relevancia: poder decir «no sé»
El reranking **siempre** devuelve 3 fragmentos, aunque la pregunta no tenga nada que ver con el corpus. Un auditor que contesta con lo que sea es peligroso. La solución es un **umbral**: si ni el mejor fragmento lo supera, **no se aprueba ni se rechaza: se escala**.

¿Cómo elegirlo? Se miden los mejores puntajes de preguntas **cubiertas** por el corpus y de preguntas **que no tienen nada que ver**. Si hay un hueco entre los dos grupos, cualquier valor dentro lo separa. Y **se recalibra** cada vez que cambia el corpus o el reranker: los puntajes de cada fuente no son comparables.

In [ ]:
# 6.1 · Calibrar el umbral
fuera_del_corpus = ["Compra de un yate para la dirección",
                    "Receta de paella valenciana para ocho personas",
                    "xyzzy qwerty plugh"]
cubiertas = [c["descripcion"] for c in casos]

def calibrar_umbral(candidatos_fn, cubiertas: list, fuera: list) -> float:
    """Mejor score tras reranking de consultas cubiertas y de consultas fuera del corpus; devuelve el umbral."""
    def mejor(q):
        top = reranker.rerank(q, candidatos_fn(q), 1)
        return (top[0].score, top[0].chunk.citation) if top else (0.0, "—")
    print("Mejor score tras reranking")
    print("  consultas CUBIERTAS:")
    rel = []
    for q in cubiertas:
        s, cita = mejor(q)
        rel.append(s)
        print(f"    {s:.3f}  {q[:46]:46} → {cita[:46]}")
    print("  consultas FUERA del corpus:")
    irr = []
    for q in fuera:
        s, _ = mejor(q)
        irr.append(s)
        print(f"    {s:.3f}  {q[:46]}")
    a, b = max(irr), min(rel)
    print(f"\nmáximo de las irrelevantes = {a:.3f}   mínimo de las relevantes = {b:.3f}")
    if a < b:
        print("→ hay un hueco: cualquier umbral dentro de ese rango separa los dos grupos")
        return round((a + b) / 2, 2)
    print("→ ⚠️ los grupos se SOLAPAN: ningún umbral los separa del todo; uso el mínimo de las relevantes")
    return round(b, 2)

UMBRAL = calibrar_umbral(recuperar_pol, cubiertas, fuera_del_corpus)
print("\nUMBRAL elegido:", UMBRAL)
q = fuera_del_corpus[0]
r = [h for h in reranker.rerank(q, recuperar_pol(q), RERANK_N) if h.score >= UMBRAL]
print(f"«{q}» → {len(r)} fragmentos tras el umbral", "→ ESCALAR" if not r else "")

**🧪 Ejercicio 6 · Elige tú el umbral.** El notebook tomó el **punto medio del hueco**. Elige otro valor **dentro** del hueco y luego uno **fuera** (muy bajo, y muy alto) y mira cuántos fragmentos pasan para las consultas **fuera del corpus** (deben quedar en cero) y para las **cubiertas** (deben conservar al menos uno). ¿Qué pasa con un umbral demasiado bajo? ¿Y con uno demasiado alto? 💸 Unas 9 llamadas al reranker por cada valor que pruebes.

In [ ]:
# 🧪 Ejercicio 6 · El umbral, a tu elección
UMBRAL_SUGERIDO = UMBRAL
UMBRAL = UMBRAL_SUGERIDO        # TODO: prueba un valor dentro del hueco, y luego uno muy bajo (0.0) y uno muy alto (0.5)
print("UMBRAL:", UMBRAL)
print("\nConsultas FUERA del corpus (deben quedar sin fragmentos):")
for q in fuera_del_corpus:
    n = sum(h.score >= UMBRAL for h in reranker.rerank(q, recuperar_pol(q), RERANK_N))
    print(f"  {n} fragmentos pasan · {q[:50]}")
print("\nConsultas CUBIERTAS (deben conservar al menos uno):")
for c in casos:
    n = sum(h.score >= UMBRAL for h in reranker.rerank(c["descripcion"], recuperar_pol(c["descripcion"]), RERANK_N))
    print(f"  {n} fragmentos pasan · {c['id']}")
print("\n(deja el umbral en un valor razonable antes de seguir: lo usan las partes siguientes)")

## Parte 7 · El auditor sobre documentos no estructurados
Juntamos todo. Una **herramienta** hace el RAG de dos etapas con el umbral y devuelve **citas**; un **agente** decide con **solo** lo que la herramienta devuelve y responde con una salida **estructurada** (`AuditDecision`: decisión, justificación y fuentes citadas). La **lista vacía** significa «no apruebes ni rechaces: escala».

In [ ]:
# 7.1 · El auditor se fabrica a partir de «de dónde salen los candidatos»
from google.adk.agents import Agent
from rag.auditor import AuditDecision, Citation
from rag.chunking import load_corpus

ARTICULOS_REALES = {c.article for c in load_corpus(POLICIES_DIR)}       # solo para detectar citas inventadas

def evidencia(consulta: str, candidatos, umbral: float) -> dict:
    """Etapa 2 y umbral: de los candidatos de cualquier fuente a las citas que ve el agente."""
    hits = [h for h in reranker.rerank(consulta, candidatos, RERANK_N) if h.score >= umbral]
    return {"query": consulta,
            "citations": [Citation(chunk_id=h.chunk.id, citation=h.chunk.citation, score=h.score,
                                   text=h.chunk.text[:1200]).model_dump() for h in hits]}

def herramienta_de(nombre: str, resolver, que_busca: str):
    """Fabrica la herramienta del agente. resolver(consulta) -> dict con las citas; se evalúa al llamar,
    así recoge un umbral recalibrado."""
    def herramienta(consulta: str) -> dict:
        return resolver(consulta)
    herramienta.__name__ = herramienta.__qualname__ = nombre
    herramienta.__doc__ = (f"Busca {que_busca} los 3 fragmentos más relevantes para la consulta.\n\n"
                           "    Args:\n        consulta: descripción del gasto o pregunta normativa.\n\n"
                           "    Returns:\n        dict con 'citations' (id, cita, score, texto). Lista vacía si nada supera el umbral:\n"
                           "        en ese caso NO se debe aprobar ni rechazar, solo escalar.\n    ")
    return herramienta

INSTRUCCION = ("Eres auditor de políticas financieras. Para cada gasto: 1) llama {herramienta}; 2) decide APROBADO o "
               "RECHAZADO solo con los fragmentos devueltos; 3) cita cada fuente con el nombre exacto que devuelve la "
               "herramienta (por ejemplo «POL-VIA-2026, Art. 1.3 (Autorización previa)»). Si la lista de citas está vacía, "
               "o los fragmentos devueltos no aplican al gasto, responde ESCALAR. Nunca uses conocimiento externo ni "
               "inventes artículos ni cifras.")
QUITAR_REGLA_ESCALAR = False      # 🧪 TODO (ejercicio 7): ponlo en True y vuelve a ejecutar esta celda y la 7.3
if QUITAR_REGLA_ESCALAR:
    INSTRUCCION = INSTRUCCION.replace("Si la lista de citas está vacía, o los fragmentos devueltos no aplican al gasto, responde ESCALAR. ", "")

def auditor_con(herramienta, nombre: str, extra: str = "") -> Agent:
    return Agent(name=nombre, model=MODELO,
                 description="Audita gastos contra políticas internas citando fuentes exactas.",
                 instruction=INSTRUCCION.format(herramienta=herramienta.__name__) + extra,
                 tools=[herramienta], output_schema=AuditDecision)

def leer_decision(texto):
    try:
        return AuditDecision.model_validate_json(texto)
    except Exception:
        return None

def cita_valida(cita: str) -> bool:
    m = re.search(r"(\d+\.\d+)", cita)
    return bool(m and m.group(1) in ARTICULOS_REALES) or "10-Q" in cita        # artículo real o fragmento del reporte

async def correr_casos(agente, prefijo: str, lista: list, mostrar=()) -> dict:
    """Pregunta al agente cada caso. Con verbose=False, preguntar() devuelve el texto (con verbose=True devuelve None)."""
    res = {}
    for c in lista:
        texto = await preguntar(agente, c["descripcion"], session_id=f"{prefijo}-{c['id']}", verbose=False)
        res[c["id"]] = leer_decision(texto)
        if c["id"] in mostrar:
            print(f"🤖 {c['id']}: {texto}\n")
    return res

def tabla_decisiones(res: dict, lista: list, **otros) -> None:
    """Decisiones del agente frente a lo esperado, y frente a otras corridas (columnas extra). Explica lo que no coincide."""
    print(f"{'caso':6} {'esperado':10} {'obtenido':10}" + "".join(f" {n:10}" for n in otros) + " citas inválidas")
    for c in lista:
        d = res.get(c["id"])
        if d is None:
            print(f"{c['id']:6} {c['esperado']:10} ⚠️ no es AuditDecision")
            continue
        malas = [x for x in d.articulos_citados if not cita_valida(x)]
        cols = "".join(f" {(o[c['id']].decision if o.get(c['id']) else '—'):10}" for o in otros.values())
        print(f"{c['id']:6} {c['esperado']:10} {d.decision:10}{cols} {malas or 'ninguna'}")
    for c in lista:                                   # si algo no coincide, muestra POR QUÉ
        d = res.get(c["id"])
        if d is not None and d.decision != c["esperado"]:
            print(f"\n⚠️ {c['id']}: esperado {c['esperado']}, obtenido {d.decision}. Justificación del agente:")
            print(f"   {d.justificacion[:600]}")
            print(f"   citas: {d.articulos_citados}")

resolver_pol = lambda consulta: evidencia(consulta, recuperar_pol(consulta), UMBRAL)
auditor_nest = auditor_con(herramienta_de("buscar_politicas", resolver_pol, "en las políticas internas (PDFs)"),
                           "auditor_politicas_pdf")
r = resolver_pol("Viaje internacional de 4 días sin autorización previa del Director")
print(len(r["citations"]), "citas")
for c in r["citations"]:
    print(f"  {c['score']:.3f}  {c['citation']}")

Primero un caso, **viendo los eventos**: 🔧 llamada a la herramienta → 📥 resultado → 🤖 decisión estructurada. 💸

In [ ]:
# 7.2 · Un caso viendo los eventos
caso_g2 = next(c for c in casos if c["id"] == "G2")
await preguntar(auditor_nest, caso_g2["descripcion"], session_id="demo-G2")

In [ ]:
# 7.3 · Los 6 casos + uno fuera del corpus
FUERA = {"id": "FUERA", "descripcion": "Compra de un yate para la dirección", "esperado": "ESCALAR"}
lista = [{"id": c["id"], "descripcion": c["descripcion"], "esperado": c["esperado"]} for c in casos] + [FUERA]
RES_NEST = await correr_casos(auditor_nest, "nest", lista)
tabla_decisiones(RES_NEST, lista)

In [ ]:
# 7.4 · ¿Lo logramos? Verificación
cand1 = candidatos_por_caso[casos[0]["id"]]
checks = {
    f"Etapa 1 devuelve candidatos (hasta K = {RETRIEVE_K})": all(0 < len(v) <= RETRIEVE_K for v in candidatos_por_caso.values()),
    f"Etapa 2 devuelve como máximo N = {RERANK_N}": len(reranker.rerank(casos[0]["descripcion"], cand1, RERANK_N)) <= RERANK_N,
    "El umbral deja vacía una consulta fuera del corpus": evidencia(fuera_del_corpus[0], recuperar_pol(fuera_del_corpus[0]), UMBRAL)["citations"] == [],
    "Las 7 respuestas del agente son AuditDecision válidas": all(d is not None for d in RES_NEST.values()),
    "Los 6 casos coinciden con lo esperado": all(RES_NEST[c["id"]] and RES_NEST[c["id"]].decision == c["esperado"] for c in casos),
    "FUERA: ESCALAR sin citas": bool(RES_NEST["FUERA"] and RES_NEST["FUERA"].decision == "ESCALAR" and not RES_NEST["FUERA"].articulos_citados),
    "Ninguna cita del agente es inventada": all(cita_valida(x) for d in RES_NEST.values() if d for x in d.articulos_citados),
}
for nombre, ok in checks.items():
    print("✅" if ok else "❌", nombre)
print("\n", "🎉 El auditor funciona sobre documentos no estructurados" if all(checks.values())
      else "Revisa las líneas con ❌: el modelo no es determinista, relee su justificación antes de concluir")

**🧪 Ejercicio 7 · ¿Basta el umbral para decir «no sé»?** En la celda 7.1 pon `QUITAR_REGLA_ESCALAR = True` (la instrucción deja de decir «si no hay citas, escala»), ejecútala de nuevo y vuelve a ejecutar la 7.3. ¿Qué responde el agente con el caso del **yate**, cuando la herramienta le devuelve una lista **vacía** pero la instrucción ya no le dice qué hacer? ¿Por qué hacen falta **las dos cosas**: el umbral y la instrucción? Al terminar, vuelve a ponerlo en `False`.

## Parte 8 · Lo mismo con documentos estructurados
Ahora el **mismo auditor** con las políticas en **markdown**, cortadas **por artículo**: un fragmento es **un** artículo, con su encabezado. Los vectores viven en memoria (la matriz de un `VertexEmbeddingRetriever`), pero el reranker, el umbral y el agente son **los mismos**. Solo cambia `recuperar_est`, la función que da los candidatos.

La pregunta es: **¿qué compra la estructura?**

In [ ]:
# 8.1 · Las políticas en markdown: un chunk por artículo, embeddings en memoria
from rag.retrieval import VertexEmbeddingRetriever
corpus_est = load_corpus(POLICIES_DIR)
retriever_est = VertexEmbeddingRetriever(corpus_est)
recuperar_est = lambda consulta: retriever_est.search(consulta, RETRIEVE_K)
print(f"{len(corpus_est)} chunks (uno por artículo) · matriz en memoria: {retriever_est.matrix.shape}\n")
h_est = tabla_hit3(recuperar_est)

In [ ]:
# 8.2 · Su propio umbral (los puntajes de cada fuente no son comparables)
UMBRAL_E = calibrar_umbral(recuperar_est, cubiertas, fuera_del_corpus)
print("\nUMBRAL_E =", UMBRAL_E)

In [ ]:
# 8.3 · El auditor sobre lo estructurado, frente al de los PDFs
resolver_est = lambda consulta: evidencia(consulta, recuperar_est(consulta), UMBRAL_E)
auditor_est = auditor_con(herramienta_de("buscar_politicas", resolver_est, "en las políticas internas (markdown, un chunk por artículo)"),
                          "auditor_politicas_md")
RES_EST = await correr_casos(auditor_est, "est", lista)
tabla_decisiones(RES_EST, lista, PDF=RES_NEST)

## Parte 9 · Combinado: políticas por artículo + reporte 10-Q en PDF 
En una empresa real, los documentos no llegan todos iguales: las **políticas** son un reglamento limpio, y el **reporte trimestral** llega como PDF. Cada uno se trata **a su manera**, y el auditor busca en **los dos a la vez**:

- Las **políticas**: cortadas por artículo (Parte 8).
- El **10-Q**: leído por el analizador de LLM (corpus `pdf_llm` de la Parte 2).

Una pregunta que **necesita las dos**: la cifra de consultoría sale del reporte (+60 %, sin aprobación del Comité) y la regla sale de las políticas (un exceso mayor a 10 % requiere al Comité Financiero).

**Un riesgo al mezclar fuentes:** si el reranker ordena todo junto, los fragmentos **largos** de una fuente pueden quedarse con **los 3 lugares** y la otra fuente nunca llega al agente. En una corrida de prueba de este notebook pasó justo eso: la pregunta mixta recibió solo fragmentos del reporte y **no vio la política**. La solución habitual es una **cuota por fuente**.

In [ ]:
# 9.1 · Candidatos de las dos fuentes y su umbral
recuperar_reporte = recuperador(c_llm)                                    # el 10-Q, leído por el analizador de LLM
recuperar_comb = lambda consulta: recuperar_est(consulta) + recuperar_reporte(consulta)
preguntas_reporte = ["¿Cuál fue la utilidad neta del 3T 2026?", "¿Qué riesgo hay por el tipo de cambio?"]
UMBRAL_C = calibrar_umbral(recuperar_comb, cubiertas + preguntas_reporte, fuera_del_corpus)
print("\nUMBRAL_C =", UMBRAL_C)

In [ ]:
# 9.2 · Cuota por fuente: UNA sola llamada al reranker, y después se reserva lugar a cada fuente
CUOTAS = {"políticas": 2, "reporte": 1}      # 🧪 TODO: prueba {"políticas": 3, "reporte": 0} y {"políticas": 1, "reporte": 2} (la suma debe ser N = 3)

def evidencia_con_cuotas(consulta: str, fuentes: dict, umbral: float, cuotas: dict) -> dict:
    # fuentes: {nombre: candidatos}. Se reordena todo junto (una llamada) y se toma de la lista, en orden,
    # hasta la cuota de cada fuente, siempre que supere el umbral. Así ninguna fuente copa los N lugares.
    todos = [h for cands in fuentes.values() for h in cands]
    origen = {id(h.chunk): nombre for nombre, cands in fuentes.items() for h in cands}
    ordenados = reranker.rerank(consulta, todos, len(todos))
    usados, elegidos = {n: 0 for n in fuentes}, []
    for h in ordenados:
        n = origen[id(h.chunk)]
        if h.score >= umbral and usados[n] < cuotas[n] and len(elegidos) < RERANK_N:
            elegidos.append(h)
            usados[n] += 1
    return {"query": consulta,
            "citations": [Citation(chunk_id=h.chunk.id, citation=h.chunk.citation, score=h.score,
                                   text=h.chunk.text[:1200]).model_dump() for h in elegidos]}

def resolver_comb(consulta: str) -> dict:
    fuentes = {"políticas": recuperar_est(consulta), "reporte": recuperar_reporte(consulta)}
    return evidencia_con_cuotas(consulta, fuentes, UMBRAL_C, CUOTAS)

MIXTA = {"id": "MIXTA", "esperado": "RECHAZADO",
         "descripcion": ("El reporte trimestral 10-Q del 3T 2026 muestra servicios de consultoría de 2,880 frente a "
                         "1,800 del año anterior. ¿Cumple ese gasto con la política?")}
q = MIXTA["descripcion"]
sin = evidencia(q, recuperar_comb(q), UMBRAL_C)["citations"]               # todo junto, sin cuotas
con = resolver_comb(q)["citations"]                                        # con cuota por fuente
print("SIN cuotas (todo junto):")
for c in sin:
    print(f"  {c['score']:.3f}  {c['citation']}")
print("\nCON cuotas (2 políticas + 1 reporte):")
for c in con:
    print(f"  {c['score']:.3f}  {c['citation']}")

In [ ]:
# 9.3 · El auditor combinado (solo cambia el «resolver»: cómo se juntan las fuentes)
auditor_comb = auditor_con(
    herramienta_de("buscar_politicas_y_reporte", resolver_comb,
                   "entre las políticas internas (por artículo) y el reporte trimestral 10-Q (PDF)"),
    "auditor_combinado",
    extra=(" Las fuentes pueden ser artículos de política o fragmentos del reporte 10-Q en PDF: cita cada una con el nombre "
           "exacto que devuelve la herramienta (por ejemplo «10-Q 3T 2026 (PDF), fragmento 2 (Item 1 – Estados financieros)»). "
           "Si una cifra del reporte contradice la política, dilo en la justificación."))
lista_c = [c for c in lista if c["id"] != "FUERA"] + [MIXTA, FUERA]
RES_COMB = await correr_casos(auditor_comb, "comb", lista_c, mostrar=("MIXTA",))
tabla_decisiones(RES_COMB, lista_c, PDF=RES_NEST, MD=RES_EST)

In [ ]:
# 9.4 · ¿Funcionó la combinación?
d_m, d_f = RES_COMB["MIXTA"], RES_COMB["FUERA"]
validas = [d for d in RES_COMB.values() if d]
checks = {
    "Las 8 respuestas son AuditDecision válidas": len(validas) == len(RES_COMB),
    "MIXTA: RECHAZADO citando una política y el reporte": bool(d_m and d_m.decision == "RECHAZADO"
        and any(re.search(r"Art", x) for x in d_m.articulos_citados) and any("10-Q" in x for x in d_m.articulos_citados)),
    "FUERA: ESCALAR sin citas": bool(d_f and d_f.decision == "ESCALAR" and not d_f.articulos_citados),
    "Ninguna cita inventada": all(cita_valida(x) for d in validas for x in d.articulos_citados),
    "Los 6 casos coinciden con lo esperado (¿el reporte los alteró?)": all(RES_COMB[c["id"]] and RES_COMB[c["id"]].decision == c["esperado"] for c in casos),
}
for nombre, ok in checks.items():
    print("✅" if ok else "❌", nombre)

**🧪 Ejercicio 9 · Las cuotas.** En la celda 9.2 cambia `CUOTAS` a `{"políticas": 3, "reporte": 0}` y ejecuta **9.2 y 9.3**. ¿Qué cita ahora el agente en la pregunta mixta? Prueba también `{"políticas": 1, "reporte": 2}`. ¿Cuándo deja de citar la política? ¿Y cuándo deja de ver el reporte? Piensa en un **enrutador** que decida a qué fuente preguntar: ¿en qué casos preferirías uno a las cuotas?

## Parte 10 · 🧹 Limpieza (ejecútala)
Borra **todos los corpus** que creó este notebook (no se puede deshacer) y el **bucket** con los PDFs. Las operaciones sobre corpus no admiten concurrencia: se borran uno tras otro.

In [ ]:
# 10 · Limpieza
for nombre in CORPUS_CREADOS:
    print(json.dumps(rest("DELETE", nombre, params={"force": "true"}), indent=2)[:160])
    time.sleep(3)
b = gcs.bucket(BUCKET)
if b.exists():
    b.delete(force=True)
print("\nlimpieza enviada: vuelve a ejecutar la celda 2.2 y confirma que ya no hay corpus")


## Lo que viene
- **Módulo 3:** el mismo RAG con reranking, pero sobre el **catálogo de datos** (esquemas y tablas) antes de generar SQL.
- **Módulo 5:** evaluar este agente con métricas de calidad RAG (*Faithfulness*, *Context Recall*) usando casos como los de hoy.
- **Capstone:** este auditor será el «Agente RAG + Reranking» del orquestador.